##### substring
- To **extract** a **portion of a string** column in a DataFrame.
- It takes **three parameters**:
  - The column containing the **string**
  - The **starting index** of the **substring (1-based)**
  - optionally, the **length of the substring**.
- It operates similarly to the **SUBSTRING()** function in **SQL**.

**Important Notes:**
- Indexing in PySpark strings is **1-based (not 0-based)**.
- If the **start index exceeds string length → result is empty**.

##### Syntax
     substring(string, start, length)

|  ARGUMENT  |  DESCRIPTION. |
|------------|---------------|
| **string** | The string to extract from |
| **start**. | The start position. The **first position** in string is **1**. This specifies that the **substring extraction** should **start** from the **1st character**. |
| **length** | The **number of characters** to extract must be a **positive number**. |
|            | This calculates the **number of characters** to extract **from the 1st position onward**. |

In [0]:
from pyspark.sql.functions import col, lit, substring, concat, length, expr

##### 1) Basic Usage

In [0]:
from pyspark.sql.functions import col

df1 = spark.createDataFrame([("hello world",)], ["text"])
display(df1)

df1.select(col("text").substr(7)).display()

text
hello world


---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
File <command-8132384556186198>, line 6
      3 df1 = spark.createDataFrame([("hello world",)], ["text"])
      4 display(df1)
----> 6 df1.select(col("text").substr(7)).display()

File /databricks/python/lib/python3.12/site-packages/pyspark/errors/utils.py:279, in _with_origin.<locals>.wrapper(*args, **kwargs)
    276 set_current_origin(func.__name__, _capture_call_site(depth))
    278 try:
--> 279     return func(*args, **kwargs)
    280 finally:
    281     set_current_origin(None, None)

TypeError: Column.substr() missing 1 required positional argument: 'length'

In [0]:
# Since substring requires a length, you simulate “till end” by giving a large number.
df1.select(substring(col("text"), 7, 1000).alias("extract")).display()

extract
world


In [0]:
# If the start index exceeds string length → result is empty.
df1.select(substring(col("text"), 12, 20).alias("extract")).display()

extract
""


In [0]:
data = [(1, "20200828", "/Date(1493596800000)/"),
        (2, "20180525", "/Date(2840054400000)/"),
        (3, "20220615", "/Date(9870064411000)/"),
        (4, "20210719", "/Date(4840072198000)/"),
        (5, "20230812", "/Date(5320012234000)/"),
        (6, "20240917", "/Date(6520038717000)/")]
        
columns = ["id", "d1", "d2"]

df = spark.createDataFrame(data, columns)
display(df)

id,d1,d2
1,20200828,/Date(1493596800000)/
2,20180525,/Date(2840054400000)/
3,20220615,/Date(9870064411000)/
4,20210719,/Date(4840072198000)/
5,20230812,/Date(5320012234000)/
6,20240917,/Date(6520038717000)/


In [0]:
# substring(col('d2'), 7, length(col('d2')) - 8) extracts the substring starting from the 7th character of 'd2'
# and takes (length of 'd2' - 8) characters. The -8 removes the prefix '/Date(' (6 chars) and suffix ')/' (2 chars).
# Example: '/Date(1493596800000)/' -> '1493596800000'

df1 = df.select("d1",
    substring(col('d1'), 1, 4).alias('year'),
    substring(col('d1'), 5, 2).alias('month'),
    substring(col('d1'), 7, 2).alias('day'),
    "d2",
    substring(col('d2'), 7, length(col('d2')) - 8).alias('trimmed_d2')  # removes '/Date(' and ')/'
)

display(df1)

d1,year,month,day,d2,trimmed_d2
20200828,2020,08,28,/Date(1493596800000)/,1493596800000
20180525,2018,05,25,/Date(2840054400000)/,2840054400000
20220615,2022,06,15,/Date(9870064411000)/,9870064411000
20210719,2021,07,19,/Date(4840072198000)/,4840072198000
20230812,2023,08,12,/Date(5320012234000)/,5320012234000
20240917,2024,09,17,/Date(6520038717000)/,6520038717000


In [0]:
df2 = df.withColumn('year', substring('d1', 1, 4))\
        .withColumn('month', substring('d1', 5, 2))\
        .withColumn('day', substring('d1', 7, 2)) \
        .withColumn("time_millis", substring('d2', 7, length('d2')-8))

display(df2)

id,d1,d2,year,month,day,time_millis
1,20200828,/Date(1493596800000)/,2020,08,28,1493596800000
2,20180525,/Date(2840054400000)/,2018,05,25,2840054400000
3,20220615,/Date(9870064411000)/,2022,06,15,9870064411000
4,20210719,/Date(4840072198000)/,2021,07,19,4840072198000
5,20230812,/Date(5320012234000)/,2023,08,12,5320012234000
6,20240917,/Date(6520038717000)/,2024,09,17,6520038717000


In [0]:
df4 = df.withColumn('year', col('d1').substr(1, 4))\
        .withColumn('month', col('d1').substr(5, 2))\
        .withColumn('day', col('d1').substr(7, 2)) \
        .withColumn("time_millis", substring(col('d2'), 7, length(col('d2'))-8))

display(df4)

id,d1,d2,year,month,day,time_millis
1,20200828,/Date(1493596800000)/,2020,08,28,1493596800000
2,20180525,/Date(2840054400000)/,2018,05,25,2840054400000
3,20220615,/Date(9870064411000)/,2022,06,15,9870064411000
4,20210719,/Date(4840072198000)/,2021,07,19,4840072198000
5,20230812,/Date(5320012234000)/,2023,08,12,5320012234000
6,20240917,/Date(6520038717000)/,2024,09,17,6520038717000


In [0]:
df3 = df.selectExpr(
    "d1",
    "substring(d1, 1, 4) as year",
    "substring(d1, 5, 2) as month",
    "substring(d1, 7, 2) as day",
    "d2",
    "substring(d2, 7, length(d2)-8) as trimmed_d2"
)

display(df3)

d1,year,month,day,d2,trimmed_d2
20200828,2020,08,28,/Date(1493596800000)/,1493596800000
20180525,2018,05,25,/Date(2840054400000)/,2840054400000
20220615,2022,06,15,/Date(9870064411000)/,9870064411000
20210719,2021,07,19,/Date(4840072198000)/,4840072198000
20230812,2023,08,12,/Date(5320012234000)/,5320012234000
20240917,2024,09,17,/Date(6520038717000)/,6520038717000


##### 2) Get `Substring` from `end of the column` in pyspark

In [0]:
df2 = spark.createDataFrame([[1, 'rush~HouR'],
                             [2, 'kung-Fu!Panda'],
                             [3, 'titaniC'],
                             [4, 'the Sixth sense']], schema="id int, title string")
display(df2)

id,title
1,rush~HouR
2,kung-Fu!Panda
3,titaniC
4,the Sixth sense


In [0]:
df3 = df2 \
  .withColumn("substring_from_end_01", substring("title", -2, 2)) \
  .withColumn("substring_from_end_02", substring("title", -6, 5))
display(df3)

id,title,substring_from_end_01,substring_from_end_02
1,rush~HouR,uR,h~Hou
2,kung-Fu!Panda,da,!Pand
3,titaniC,iC,itani
4,the Sixth sense,se,sens


##### 3) Extract `N characters` from string column in pyspark

In [0]:
df4 = df2.withColumn('new_string', concat(substring("title", 1, 3), lit('_'), substring("title", 6, 2)))
display(df4)

id,title,new_string
1,rush~HouR,rus_Ho
2,kung-Fu!Panda,kun_Fu
3,titaniC,tit_iC
4,the Sixth sense,the_ix
